# UnifyWeaver를 활용한 가계도 튜토리얼

이 인터랙티브 노트북은 UnifyWeaver를 사용하여 Prolog 서술어를 Bash 스크립트로 컴파일하는 방법을 시연합니다.

## 사전 요구 사항

- SWI-Prolog 설치
- UnifyWeaver 라이브러리 사용 가능
- Prolog Jupyter 커널 설치 (`pip install prolog-jupyter-kernel`)

## 학습 목표

이 노트북을 마치면 다음을 수행할 수 있습니다:
1. Prolog 사실 및 규칙 정의
2. UnifyWeaver를 사용하여 서술어를 Bash로 컴파일
3. 생성된 Bash 스크립트 테스트
4. 이행 폐포 컴파일 메커니즘 이해

## 1단계: UnifyWeaver 환경 초기화

먼저 UnifyWeaver 모듈을 로드해야 합니다. education 디렉터리의 `init.pl` 파일을 사용합니다.

In [ ]:
% Load the initialization file
['../init'].

## 2단계: 가족 관계 정의

성경 속 가계도로부터 부모-자식 관계를 정의해 보겠습니다.

In [ ]:
% Define parent facts
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## 3단계: 부모 쿼리 테스트

컴파일하기 전에 몇 가지 Prolog 쿼리로 데이터가 올바른지 확인합니다.

In [ ]:
% Query: Who are Abraham's children?
parent(abraham, Child).

In [ ]:
% Query: Who are Jacob's children?
parent(jacob, Child).

## 4단계: 조상 관계 정의

이제 이행 폐포인 `ancestor` 관계를 정의해 보겠습니다.

In [ ]:
% Define ancestor as transitive closure of parent
:- dynamic ancestor/2.

% Base case: parent is an ancestor
ancestor(X, Y) :- parent(X, Y).

% Recursive case: if X is parent of Y and Y is ancestor of Z, then X is ancestor of Z
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## 5단계: 조상 쿼리 테스트

ancestor 서술어가 올바르게 동작하는지 확인합니다.

In [ ]:
% Query: Is Abraham an ancestor of Jacob?
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% Query: Who are all of Abraham's descendants?
ancestor(abraham, Descendant).

## 6단계: 부모 사실을 Bash로 컴파일

이제 흥미로운 부분입니다 — `parent/2` 사실을 Bash 스크립트로 컴파일해 보겠습니다!

In [ ]:
% Load the stream compiler
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % Compile parent facts to bash
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## 7단계: 부모 스크립트 저장

생성된 Bash 코드를 파일에 저장합니다.

In [ ]:
% Save to file
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## 8단계: 조상 서술어를 Bash로 컴파일

이제 재귀를 사용하는 `ancestor/2` 서술어를 컴파일합니다.

In [ ]:
% Load the recursive compiler
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % Compile ancestor to bash
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## 9단계: 조상 스크립트 저장

ancestor 스크립트를 파일에 저장합니다.

In [ ]:
% Save to file
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## 10단계: 생성된 스크립트 테스트

이제 생성된 Bash 스크립트를 테스트해 보겠습니다! `%%bash` 매직 명령을 사용하여 bash 명령을 실행합니다.

In [ ]:
%%bash
# Source the parent script
source ../output/parent.sh

# Test: Who are Abraham's children?
echo "Abraham's children:"
parent abraham

In [ ]:
%%bash
# Source both scripts
source ../output/parent.sh
source ../output/ancestor.sh

# Test: Who are Abraham's descendants?
echo "Abraham's descendants:"
ancestor abraham

In [ ]:
%%bash
# Source both scripts
source ../output/parent.sh
source ../output/ancestor.sh

# Test: Is Abraham an ancestor of Judah?
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ Yes, Abraham is an ancestor of Judah"
else
    echo "✗ No"
fi

## 11단계: 컴파일 전략 이해하기

UnifyWeaver가 수행한 작업을 분석해 보겠습니다:

1. **parent 컴파일**: `stream_compiler`를 사용하여 모든 부모-자식 쌍을 출력하는 간단한 스트리밍 함수를 생성했습니다.

2. **ancestor 컴파일**: 이행 폐포 패턴을 감지하고 너비 우선 탐색(BFS) 최적화를 적용하여 도달 가능한 모든 조상을 효율적으로 계산했습니다.

컴파일 전략을 확인해 보겠습니다:

In [ ]:
% Check if ancestor is classified as recursive
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## 요약

이 노트북에서 배운 내용:

✅ Prolog 사실 및 규칙 정의 방법

✅ 사실에 대해 UnifyWeaver의 `stream_compiler`를 사용하는 방법

✅ 재귀적 서술어에 대해 UnifyWeaver의 `recursive_compiler`를 사용하는 방법

✅ 생성된 Bash 스크립트를 테스트하는 방법

✅ UnifyWeaver가 이행 폐포를 자동으로 감지하고 BFS 최적화를 적용한다는 점

## 다음 단계

다음 연습 과제를 시도해 보세요:

1. 가계도에 가족 구성원을 더 추가하기
2. `grandparent/2` (조부모) 서술어를 정의하고 컴파일하기
3. `sibling/2` (부모가 같은 두 사람) 서술어 만들기
4. 생성된 Bash 코드를 살펴보고 BFS 알고리즘 동작 방식 파악하기

고급 재귀 패턴을 배우려면 **노트북 2: 재귀 패턴 비교**로 이동하세요!